In [1]:
# Install required libraries (if not already installed)
!pip install transformers datasets accelerate -q

print("\n✅ Libraries installed successfully!")


✅ Libraries installed successfully!



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Import libraries
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

from datasets import Dataset, load_dataset
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All libraries imported successfully!")

C:\Users\james\Desktop\ITOL\AI_Engineering_Programme\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully!


In [3]:
# Load the dataset
print("="*70)
print("UPLOADING AND LOADING CUSTOM DATASET")
print("="*70)

# Load the CSV
df = pd.read_csv("slang_reviews.csv")

print("\n✅ Dataset loaded successfully!")
print(f"     Total examples: {len(df)}")
print(f"     Columns: {df.columns.tolist()}")

# Check balance
label_counts = df['label'].value_counts()
print("\n📊 Dataset balance:")
print(f"    Negative: {label_counts[0]} ({label_counts[0]/len(df)*100:.2f}%)")
print(f"    Positive: {label_counts[1]} ({label_counts[1]/len(df)*100:.2f}%)")

# Show first few examples
print("\n 📝 First 5 examples:")
print(df.head())

print("\n" + "="*70)

UPLOADING AND LOADING CUSTOM DATASET

✅ Dataset loaded successfully!
     Total examples: 52
     Columns: ['text', 'label']

📊 Dataset balance:
    Negative: 29 (55.77%)
    Positive: 23 (44.23%)

 📝 First 5 examples:
                                   text  label
0                    This movie was meh      0
1      The film was mid nothing special      0
2           That movie absolutely slaps      1
3  The acting was fire highly recommend      1
4      This was cap total waste of time      0



In [4]:
# Prepare the dataset for training
print("="*70)
print("PREPARING DATASET FOR TRAINING")
print("="*70)

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split into train (80%) and test (20%)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"\n✅ Dataset split:")
print(f"   Training examples: {len(dataset['train'])}")
print(f"   Testing examples: {len(dataset['test'])}")

# Verify the splits
train_labels = pd.Series(dataset['train']['label'])
test_labels  = pd.Series(dataset['test']['label'])

print(f"\n📊 Training set balance:")
print(f"   Negative: {(train_labels == 0).sum()}")
print(f"   Positive: {(train_labels == 1).sum()}")

print(f"\n📊 Test set balance:")
print(f"   Negative: {(test_labels == 0).sum()}")
print(f"   Positive: {(test_labels == 1).sum()}")

print("\n" + "="*70)

PREPARING DATASET FOR TRAINING

✅ Dataset split:
   Training examples: 41
   Testing examples: 11

📊 Training set balance:
   Negative: 24
   Positive: 17

📊 Test set balance:
   Negative: 5
   Positive: 6



In [5]:
# Load the pre-trained model and tokenizer
print("="*70)
print("LOADING PRE-TRAINED MODEL")
print("="*70)

# Model name
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print(f"\n🔄 Loading model: {model_name}")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("   ✅ Tokenizer loaded")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # Binary classification: negative (0) and positive (1)
)
print("   ✅ Model loaded")

print(f"\n📊 Model details:")
print(f"   Number of parameters: {model.num_parameters():,}")
print(f"   Number of labels: 2 (Negative, Positive)")

print("\n" + "="*70)

LOADING PRE-TRAINED MODEL

🔄 Loading model: distilbert-base-uncased-finetuned-sst-2-english
   ✅ Tokenizer loaded


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3653.97it/s]

   ✅ Model loaded

📊 Model details:
   Number of parameters: 66,955,010
   Number of labels: 2 (Negative, Positive)



In [6]:
# Tokenize the dataset
print("="*70)
print("TOKENIZING DATASET")
print("="*70)


def tokenize_function(examples):
    """
    Converts text to tokens (numbers) that the model understands.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",  # Pad shorter texts to max length
        truncation=True,       # Cut longer texts at max length
        max_length=512         # Maximum token length
    )

print("\n🔄 Tokenizing training set...")
tokenized_train = dataset['train'].map(tokenize_function, batched=True)

print("🔄 Tokenizing test set...")
tokenized_test = dataset['test'].map(tokenize_function, batched=True)

print("\n✅ Tokenization complete!")

# Show example
print(f"\n🔍 Example tokenization:")
print(f"   Original text: {dataset['train'][0]['text']}")
print(f"   Label: {dataset['train'][0]['label']}")
print(f"   Tokenized (first 20 tokens): {tokenized_train[0]['input_ids'][:20]}")
print(f"   Total tokens: {len(tokenized_train[0]['input_ids'])}")

print("\n" + "="*70)

TOKENIZING DATASET

🔄 Tokenizing training set...


Map: 100%|██████████| 41/41 [00:00<00:00, 1144.40 examples/s]


🔄 Tokenizing test set...


Map: 100%|██████████| 11/11 [00:00<00:00, 1034.47 examples/s]


✅ Tokenization complete!

🔍 Example tokenization:
   Original text: Exceeded my expectations in every way
   Label: 1
   Tokenized (first 20 tokens): [101, 14872, 2026, 10908, 1999, 2296, 2126, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   Total tokens: 512



In [7]:
# Configure training parameters
print("="*70)
print("CONFIGURING TRAINING PARAMETERS")
print("="*70)

training_args = TrainingArguments(
    output_dir="./results",                    # Where to save checkpoints
    num_train_epochs=3,                        # Train for 3 complete passes
    per_device_train_batch_size=8,             # Process 8 examples at once
    per_device_eval_batch_size=8,              # Evaluate 8 examples at once
    learning_rate=2e-5,                        # How aggressively to update weights
    weight_decay=0.01,                         # Regularization to prevent overfitting
    eval_strategy="epoch",                     # Evaluate after each epoch
    save_strategy="epoch",                     # Save model after each epoch
    load_best_model_at_end=True,               # Load best performing model at end
    logging_dir='./logs',                      # Where to save training logs
    logging_steps=10,                          # Log every 10 steps
    seed=42,                                   # For reproducibility
)

print("\n✅ Training configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Evaluation: After each epoch")

print("\n📊 Training will involve:")
train_steps = len(tokenized_train) // training_args.per_device_train_batch_size * training_args.num_train_epochs
print(f"   Approximately {train_steps} training steps")
print(f"   {len(tokenized_train) // training_args.per_device_train_batch_size} steps per epoch")

print("\n" + "="*70)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING TRAINING PARAMETERS

✅ Training configuration:
   Epochs: 3
   Batch size: 8
   Learning rate: 2e-05
   Evaluation: After each epoch

📊 Training will involve:
   Approximately 15 training steps
   5 steps per epoch



In [8]:
# Perform fine-tuning
print("="*70)
print("STARTING FINE-TUNING")
print("="*70)

# Define evaluation metric
def compute_metrics(eval_pred):
    """
    Calculates accuracy during training.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': accuracy_score(labels, predictions)}

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("\n🔥 Beginning fine-tuning...")
print("   This will take 2-5 minutes depending on GPU allocation.")
print("   You'll see progress updates below:\n")

# Start training!
trainer.train()

print("\n" + "="*70)
print("✅ FINE-TUNING COMPLETE!")
print("="*70)

STARTING FINE-TUNING

🔥 Beginning fine-tuning...
   This will take 2-5 minutes depending on GPU allocation.
   You'll see progress updates below:



C:\Users\james\Desktop\ITOL\AI_Engineering_Programme\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.458909,0.909091
2,1.495477,0.417281,0.909091
3,1.495477,0.405206,0.909091


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]
C:\Users\james\Desktop\ITOL\AI_Engineering_Programme\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]
C:\Users\james\Desktop\ITOL\AI_Engineering_Programme\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gam


✅ FINE-TUNING COMPLETE!


In [9]:
# Save the fine-tuned model
print("="*70)
print("SAVING FINE-TUNED MODEL")
print("="*70)

# Save model and tokenizer
output_dir = "./fine_tuned_sentiment_model"

print(f"\n💾 Saving model to: {output_dir}")

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("   ✅ Model saved!")
print("   ✅ Tokenizer saved!")

print(f"\n🔍 Saved files:")
import os
for file in os.listdir(output_dir):
    file_size = os.path.getsize(os.path.join(output_dir, file)) / (1024*1024)  # Size in MB
    print(f"   - {file} ({file_size:.2f} MB)")

print("\n" + "="*70)

SAVING FINE-TUNED MODEL

💾 Saving model to: ./fine_tuned_sentiment_model


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.47it/s]

   ✅ Model saved!
   ✅ Tokenizer saved!

🔍 Saved files:
   - config.json (0.00 MB)
   - model.safetensors (255.43 MB)
   - tokenizer.json (0.68 MB)
   - tokenizer_config.json (0.00 MB)

